# Latency Anomaly Model (MLflow)

Small MLflow extension for the Capstone_6 telecom project (handbook Day 5, "small MLflow/Genie extension"). It trains an unsupervised anomaly detector on network KPI measurements, tracks it in MLflow, registers it in Unity Catalog and writes anomaly scores to a Gold table.

**Data:** `capstone6_telecom.gold.fact_network_kpi` (one row per KPI measurement, 2026-07-01 to 2026-08-27).

**Method**
* Features: `latency_ms`, `packet_loss_pct`, `throughput_mbps`, `active_sessions`, `availability_pct`, hour of day (from `measurement_ts`), and one-hot `network_generation` and `site_type`.
* Pipeline: median imputation + standard scaling (numeric), most-frequent imputation + one-hot encoding (categorical), then `IsolationForest` (contamination 0.01, 200 trees, max_samples 2048, random_state 42). All preprocessing lives inside the logged model.
* MLflow: params, metrics and the model (with signature and input example) are logged, the model is registered as `capstone6_telecom.gold.latency_anomaly_model` and given the alias `champion`.
* Scoring: the `champion` model scores every row into `capstone6_telecom.gold.kpi_anomaly_scores` (overwrite, so re-runs are idempotent).

**About the label:** `is_iqr_outlier` is a rule-based flag from Silver DQ (Tukey IQR 1.5 on latency itself). It is not a ground-truth label, so it is used **only for evaluation** (precision / recall / F1), never as a training target.

In [0]:
import mlflow
import numpy as np
import pandas as pd
from datetime import datetime, timezone
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SOURCE_TABLE = "capstone6_telecom.gold.fact_network_kpi"
SCORES_TABLE = "capstone6_telecom.gold.kpi_anomaly_scores"
MODEL_NAME = "capstone6_telecom.gold.latency_anomaly_model"
MODEL_ALIAS = "champion"
EXPERIMENT_PATH = "/Users/datauser12@financecloudbrick.onmicrosoft.com/capstone6_latency_anomaly"

NUMERIC_FEATURES = ["latency_ms", "packet_loss_pct", "throughput_mbps", "active_sessions", "availability_pct", "hour_of_day"]
CATEGORICAL_FEATURES = ["network_generation", "site_type"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

# max_samples=2048: with the default (256 rows per tree) most trees never see one of the rare
# extreme latency readings (26 of 11,829), so they are not isolated reliably.
PARAMS = {"contamination": 0.01, "n_estimators": 200, "max_samples": 2048, "random_state": 42}

mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(EXPERIMENT_PATH)

In [0]:
pdf = spark.table(SOURCE_TABLE).select(
    "measurement_id", "tower_id", "cell_id", "region_id", "measurement_ts",
    "latency_ms", "packet_loss_pct", "throughput_mbps", "active_sessions", "availability_pct",
    "network_generation", "site_type", "is_iqr_outlier",
).toPandas()

def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """Raw model input: numeric KPIs as float, hour of day, and categorical columns as string."""
    X = pd.DataFrame(index=df.index)
    for c in ["latency_ms", "packet_loss_pct", "throughput_mbps", "active_sessions", "availability_pct"]:
        X[c] = pd.to_numeric(df[c], errors="coerce").astype("float64")
    X["hour_of_day"] = pd.to_datetime(df["measurement_ts"]).dt.hour.astype("float64")
    for c in CATEGORICAL_FEATURES:
        X[c] = df[c].astype("object").where(df[c].notna(), None)
    return X[FEATURES]

X = build_features(pdf)
y_iqr = pdf["is_iqr_outlier"].fillna(False).astype(bool).to_numpy()

print(f"Rows: {len(pdf):,}  |  IQR outliers: {y_iqr.sum()}  |  date range: {pdf['measurement_ts'].min()} .. {pdf['measurement_ts'].max()}")
print("\nNulls per feature:")
print(X.isna().sum().to_string())
display(X.describe(include="all").T.reset_index().rename(columns={"index": "feature"}))

In [0]:
preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUMERIC_FEATURES),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), CATEGORICAL_FEATURES),
])
model = Pipeline([("preprocess", preprocess), ("iforest", IsolationForest(**PARAMS))])
model.fit(X)

# IsolationForest: predict -1 = anomaly, 1 = normal
pred_anomaly = model.predict(X) == -1
tp = int((pred_anomaly & y_iqr).sum())
fp = int((pred_anomaly & ~y_iqr).sum())
fn = int((~pred_anomaly & y_iqr).sum())
precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

metrics = {
    "n_rows": len(X),
    "anomaly_count": int(pred_anomaly.sum()),
    "anomaly_rate": float(pred_anomaly.mean()),
    "iqr_outlier_count": int(y_iqr.sum()),
    "precision_vs_iqr": precision,
    "recall_vs_iqr": recall,
    "f1_vs_iqr": f1,
    "anomalies_not_iqr_outlier": fp,
}

with mlflow.start_run(run_name="isolation_forest_latency_anomaly") as run:
    mlflow.set_tags({"project": "capstone6_telecom", "handbook_day": "5", "source_table": SOURCE_TABLE})
    mlflow.log_params({**PARAMS, "model": "IsolationForest", "numeric_imputation": "median",
                       "features": ",".join(FEATURES)})
    mlflow.log_metrics(metrics)
    sample = X.head(100)
    signature = infer_signature(sample, model.predict(sample))
    model_info = mlflow.sklearn.log_model(
        model, name="model", signature=signature, input_example=X.head(3),
        registered_model_name=MODEL_NAME,
    )

client = MlflowClient()
model_version = model_info.registered_model_version
client.set_registered_model_alias(MODEL_NAME, MODEL_ALIAS, model_version)

print(f"Run: {run.info.run_id}")
print(f"Registered {MODEL_NAME} version {model_version} with alias @{MODEL_ALIAS}")
display(pd.DataFrame([metrics]).T.reset_index().rename(columns={"index": "metric", 0: "value"}))

In [0]:
champion_uri = f"models:/{MODEL_NAME}@{MODEL_ALIAS}"
champion = mlflow.sklearn.load_model(champion_uri)
champion_version = int(MlflowClient().get_model_version_by_alias(MODEL_NAME, MODEL_ALIAS).version)

X_all = build_features(pdf)
# Higher anomaly_score = more anomalous (negated IsolationForest score_samples)
scores = pd.DataFrame({
    "measurement_id": pdf["measurement_id"],
    "tower_id": pdf["tower_id"],
    "region_id": pdf["region_id"],
    "measurement_ts": pd.to_datetime(pdf["measurement_ts"]),
    "latency_ms": X_all["latency_ms"],
    "anomaly_score": -champion.score_samples(X_all),
    "is_anomaly": champion.predict(X_all) == -1,
    "is_iqr_outlier": pdf["is_iqr_outlier"].fillna(False).astype(bool),
    "model_version": champion_version,
    "scored_at": datetime.now(timezone.utc).replace(tzinfo=None),
})

(spark.createDataFrame(scores)
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(SCORES_TABLE))

print(f"Scored {len(scores):,} rows with {champion_uri} (version {champion_version}) -> {SCORES_TABLE}")

In [0]:
from pyspark.sql import functions as F

scored = spark.table(SCORES_TABLE)
source_rows = spark.table(SOURCE_TABLE).count()
scored_rows = scored.count()
distinct_ids = scored.select("measurement_id").distinct().count()

recall_row = scored.agg(
    F.sum(F.col("is_iqr_outlier").cast("int")).alias("iqr_outliers"),
    F.sum((F.col("is_iqr_outlier") & F.col("is_anomaly")).cast("int")).alias("iqr_outliers_flagged"),
    F.sum(F.col("is_anomaly").cast("int")).alias("anomalies"),
).first()

print(f"Scores table rows: {scored_rows:,} (source {source_rows:,}, distinct measurement_id {distinct_ids:,})")
print(f"Anomalies flagged: {recall_row.anomalies}")
print(f"IQR outliers flagged: {recall_row.iqr_outliers_flagged} / {recall_row.iqr_outliers}")
assert scored_rows == source_rows == distinct_ids, "Scores table row count does not match fact_network_kpi"

kpi = spark.table(SOURCE_TABLE).select("measurement_id", "network_generation")
display(
    scored.join(kpi, "measurement_id")
    .groupBy("region_id", "network_generation")
    .agg(F.sum(F.col("is_anomaly").cast("int")).alias("anomalies"),
         F.count("*").alias("measurements"),
         F.sum(F.col("is_iqr_outlier").cast("int")).alias("iqr_outliers"))
    .withColumn("anomaly_rate", F.round(F.col("anomalies") / F.col("measurements"), 4))
    .orderBy(F.desc("anomalies"), "region_id")
    .limit(10)
)